This notebook is organised into three sections: (1) data preparation, (2) fire-risk calculation, and (3) optimisation modelling.


In [ ]:
import json
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd

from shapely.ops import unary_union
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
import pulp

# 1. Data

This section loads and prepares the datasets used in the study. 

## 1.1 London boroughs

We load the boundaries of all boroughs in Greater London.


In [ ]:
borough = gpd.read_file("data/borough/London_Borough_Excluding_MHW.shp")
print(borough.head())

## 1.2 Current stations

We load a JSON file containing all London fire stations, including both operational and closed stations. We keep only the currently operational stations and convert them into point geometries (GeoDataFrame, EPSG:27700).


In [ ]:
with open("data/current_station/all_station.json", "r", encoding="utf-8") as f:
    all_station_json = json.load(f)

# convert to DataFrame and keep current stations
all_station_df = pd.DataFrame(all_station_json)
current_station_df = all_station_df.head(103) # first 103 stations are operational

# create GeoDataFrame in WGS84, then convert to EPSG:27700
current_station_gdf = gpd.GeoDataFrame(
    current_station_df,
    geometry=gpd.points_from_xy(current_station_df["longitude"], current_station_df["latitude"]),
    crs="EPSG:4326",
)
current_station = current_station_gdf.to_crs(epsg=27700)

print(current_station.head())

## 1.3 Location of fire incidents

The downloaded fire incident data come as two files: `fire_2009_17.csv` (2009–2017) and `fire_2018_onwards.xlsx` (2018 onwards). Because these raw files are large, we preprocessed them offline and saved only the cleaned table `fire_2015_2024.csv` in this repository. The preprocessing keeps calendar years 2015–2024, retains confirmed fire events (because the raw records also include false alarms and other non-fire incidents) with valid coordinates, and merges the two sources. The notebook then starts from this prepared dataset and converts the records into point geometries (GeoDataFrame, EPSG:27700).

The preprocessing code is shown below for reference only (not executed here).

```python
columns_to_keep = ["IncidentNumber", "CalYear", "IncidentGroup", "Latitude", "Longitude"]

fire_incident_1 = pd.read_csv(
    "data/fire_incident/fire_2009_17.csv",
    usecols=columns_to_keep,
    low_memory=False,
)
fire_incident_2 = pd.read_excel(
    "data/fire_incident/fire_2018_onwards.xlsx",
    usecols=columns_to_keep,
)

# keep 2015, 2016, 2017 in first dataset
fire_incident_1 = fire_incident_1[fire_incident_1["CalYear"].isin([2015, 2016, 2017])]

# in column "IncidentGroup", keep "Fire" only
fire_incident_1 = fire_incident_1[fire_incident_1["IncidentGroup"] == "Fire"]
fire_incident_2 = fire_incident_2[fire_incident_2["IncidentGroup"] == "Fire"]

# combine both datasets and drop missing coordinates
fire_incident_df = pd.concat([fire_incident_1, fire_incident_2], ignore_index=True).dropna()
fire_incident_df.to_csv("data/fire_incident/fire_2015_2024.csv", index=False)
```


In [ ]:
fire_incident_df = pd.read_csv("data/fire_incident/fire_2015_2024.csv")

# create GeoDataFrame in WGS84, then convert to EPSG:27700
fire_incident_gdf = gpd.GeoDataFrame(
    fire_incident_df,
    geometry=gpd.points_from_xy(fire_incident_df["Longitude"], fire_incident_df["Latitude"]),
    crs="EPSG:4326",
)
fire_incident = fire_incident_gdf.to_crs(epsg=27700)

print(fire_incident.head())

## 1.4 OA (centroid = demand points)

We load London OA boundaries and extract their centroids as demand points.


In [ ]:
oa = gpd.read_file("data/oa/OA_2011_London_gen_MHW.shp")
oa = oa.to_crs(epsg=27700)

# extract centroids as demand points
demand = oa.copy()
demand["geometry"] = demand.geometry.centroid

print(demand.head())
print(f"Unique OA / demand points: {demand['OA11CD'].nunique()}")

## 1.5 MSOA (centroid = new candidate stations)

We load London MSOA boundaries and extract their centroids as new candidate stations.


In [ ]:
msoa = gpd.read_file("data/msoa/MSOA_2011_London_gen_MHW.shp")
msoa = msoa.to_crs(epsg=27700)

# extract centroids as candidate stations
candidate_station = msoa.copy()
candidate_station["geometry"] = candidate_station.geometry.centroid

print(candidate_station.head())
print(f"Unique MSOA / candidate stations: {candidate_station['MSOA11CD'].nunique()}")



# 2. Fire risk

In this section we calculate fire risk for each OA. 

## 2.1 Calculation

We define fire risk as the number of fire incidents per 1,000 household residents in each OA. The resulting values are attached to OA centroids as demand weights.


In [ ]:
# calculate fire risk on OA areas
fire_in_grid = gpd.sjoin(fire_incident, oa, how="inner", predicate="within")
fire_count = fire_in_grid.groupby("OA11CD").size().reset_index(name="fire_incidents")

# write fire counts and risk back to OA
for col in ["fire_incidents", "fire_risk"]:
    if col in oa.columns:
        oa = oa.drop(columns=[col])

oa = oa.merge(fire_count, on="OA11CD", how="left")
oa["fire_incidents"] = oa["fire_incidents"].fillna(0)
oa["fire_risk"] = np.where(
    oa["HHOLDRES"] > 0,
    (oa["fire_incidents"] / oa["HHOLDRES"]) * 1000,
    0,
)

# attach fire risk weights to demand points
if "fire_risk" in demand.columns:
    demand = demand.drop(columns=["fire_risk"])
demand = demand.merge(oa[["OA11CD", "fire_risk"]], on="OA11CD", how="left")

print(oa[["OA11CD", "fire_incidents", "HHOLDRES", "fire_risk"]].head())
print(f"OA with fire risk: {len(oa)}")

## 2.2 Visualisation

We map fire risk, current stations, and their 3 km service coverage across Greater London.


In [ ]:
current_station["coverage"] = current_station.geometry.buffer(3000)  # 3km radius
colors = ["#ffffff", "#fde6ec", "#f9ccd8", "#f4aebf", "#ed87a4", "#e55a87", "#ce306a", "#b11455", "#88073f", "#5a0028"]
cmap = LinearSegmentedColormap.from_list("custom_cmap", colors, N=100)
combined_coverage = unary_union(current_station["coverage"])  # 合并所有缓冲区，生成联合区域
combined_gdf = gpd.GeoDataFrame(geometry=[combined_coverage], crs=current_station.crs)


def add_north_arrow(ax, x=0.92, y=0.92, length=0.06, color="black"):
    """
    :param x: arrow location x
    :param y: arrow location y
    :param length: arrow length
    :param color: arrow color
    """
    ax.annotate(
        "N", xy=(x, y), xytext=(x, y - length),
        arrowprops=dict(facecolor=color, width=1.5, headwidth=5, headlength=7),
        ha="center", va="center", fontsize=6, color=color,
        xycoords="axes fraction",
        annotation_clip=False,
        clip_on=False,
    )


def make_fire_risk_axes():
    fig, ax = plt.subplots(figsize=(7.2, 6), dpi=400)
    fig.subplots_adjust(left=0.02, right=0.98, bottom=0.02, top=0.98)

    borough.boundary.plot(
        edgecolor="black", linewidth=0.45, alpha=0.45, label="Borough Boundary", ax=ax
    )
    oa.plot(
        column="fire_risk",
        cmap=cmap,
        legend=False,
        alpha=0.6,
        ax=ax,
        legend_kwds={"label": "Fire Risk"},
    )
    combined_gdf.plot(
        edgecolor="black", facecolor="none", linestyle="--", linewidth=0.7, alpha=0.7, ax=ax
    )
    current_station.plot(
        color="#9ACD32", markersize=6, label="Current Station", ax=ax
    )
    return fig, ax


def finish_fire_risk_map(fig, ax, extra_legend_handles=None, save_path=None, scalebar_km=5):
    ax.axis("off")
    ax.autoscale(enable=False)

    # fill figure aspect (7.2 x 6) so saved maps share size and avoid letterbox margins
    xmin, xmax = ax.get_xlim()
    ymin, ymax = ax.get_ylim()
    data_w, data_h = xmax - xmin, ymax - ymin
    target_aspect = 7.2 / 6
    if data_w / data_h > target_aspect:
        new_h = data_w / target_aspect
        cy = 0.5 * (ymin + ymax)
        ax.set_ylim(cy - new_h / 2, cy + new_h / 2)
    else:
        new_w = data_h * target_aspect
        cx = 0.5 * (xmin + xmax)
        ax.set_xlim(cx - new_w / 2, cx + new_w / 2)

    map_xlim, map_ylim = ax.get_xlim(), ax.get_ylim()
    map_pos = ax.get_position()

    add_north_arrow(ax)

    fig.canvas.draw()
    pad_x, pad_y = 0.04, 0.04  # lower left of the axes
    x_left, y_bottom = ax.transData.inverted().transform(ax.transAxes.transform((pad_x, pad_y)))
    # fixed ground length (metres in EPSG:27700)
    scalebar_length = scalebar_km * 1000.0
    # tick height in axes fraction so London / Hillingdon look the same
    _, y_tick = ax.transData.inverted().transform(ax.transAxes.transform((pad_x, pad_y + 0.012)))
    bar_height = abs(y_tick - y_bottom)
    start_x = x_left
    start_y = y_bottom + 2 * bar_height  # keep labels inside the axes

    n_div = 5
    ax.plot(
        [start_x, start_x + scalebar_length],
        [start_y, start_y],
        color="black", linewidth=2, scalex=False, scaley=False,
    )
    for i in range(n_div + 1):
        x_tick = start_x + scalebar_length * i / n_div
        label_km = scalebar_km * i / n_div
        ax.plot(
            [x_tick, x_tick],
            [start_y, start_y - bar_height],
            color="black", linewidth=1, scalex=False, scaley=False,
        )
        ax.text(
            x_tick, start_y - 2 * bar_height,
            f"{label_km:.0f}", ha="center", va="top", fontsize=6,
        )
    ax.text(
        start_x + scalebar_length + 0.1 * scalebar_length, start_y - 2 * bar_height,
        "km", ha="left", va="top", fontsize=6,
    )

    sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=oa["fire_risk"].min(), vmax=oa["fire_risk"].max()))
    sm._A = []
    cbar_x, cbar_y, cbar_w, cbar_h = 0.92, 0.15, 0.025, 0.35
    x0, y0 = ax.transAxes.transform((cbar_x - cbar_w / 2, cbar_y))
    x1, y1 = ax.transAxes.transform((cbar_x + cbar_w / 2, cbar_y + cbar_h))
    fx0, fy0 = fig.transFigure.inverted().transform((x0, y0))
    fx1, fy1 = fig.transFigure.inverted().transform((x1, y1))
    cax = fig.add_axes([fx0, fy0, fx1 - fx0, fy1 - fy0])
    cbar = fig.colorbar(sm, cax=cax)
    cbar.set_label("Fire Risk (Incidents per 1000 people)", fontsize=6)
    cbar.ax.tick_params(labelsize=6)

    custom_lines = [
        Line2D([0], [0], color="#9ACD32", marker="o", linestyle="None", markersize=3, label="Current Fire Station"),
        Line2D([0], [0], color="black", lw=0.5, label="Borough Boundary"),
    ]
    if extra_legend_handles:
        custom_lines.extend(extra_legend_handles)
    custom_lines.append(Line2D([0], [0], color="black", linestyle="--", lw=0.5, label="Service Coverage"))
    ax.legend(handles=custom_lines, loc="lower right", fontsize=6)

    ax.set_position(map_pos)
    ax.set_xlim(map_xlim)
    ax.set_ylim(map_ylim)

    if save_path is not None:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        fig.savefig(save_path, dpi=400)
    plt.show()


fig, ax = make_fire_risk_axes()
finish_fire_risk_map(fig, ax)


The London Borough of Hillingdon is selected as the study area for the subsequent optimisation because it has high fire risk and is not fully covered by the current stations.

In [ ]:
fig, ax = make_fire_risk_axes()

hillingdon = borough[borough["GSS_CODE"] == "E09000017"]
hillingdon.boundary.plot(edgecolor="red", linewidth=0.75, ax=ax)

finish_fire_risk_map(
    fig,
    ax,
    extra_legend_handles=[Line2D([0], [0], color="red", lw=0.5, label="Study Area")],
    save_path="figure/study_area.png",
)

# 3. Modelling


## 3.1 Study-area layers

Because the optimisation focuses on Hillingdon, we clip the London-wide layers from Section 1 to this borough: demand points (OA centroids), new candidate stations (MSOA centroids), and current stations.

The **total candidate list** used in the model is then formed by placing current stations first, followed by the MSOA candidate stations.


In [ ]:
hillingdon = borough[borough["GSS_CODE"] == "E09000017"].copy()
hillingdon_geom = unary_union(hillingdon.geometry)

demand_hillingdon = demand[demand.geometry.within(hillingdon_geom)].copy()
candidate_station_hillingdon = candidate_station[
    candidate_station.geometry.within(hillingdon_geom)
].copy()
current_station_hillingdon = current_station[
    current_station.geometry.within(hillingdon_geom.buffer(50))
].copy().reset_index(drop=True)

# total candidate stations = existing stations first, then MSOA centroids
total_station_hillingdon = gpd.GeoDataFrame(
    pd.concat(
        [current_station_hillingdon[["geometry"]], candidate_station_hillingdon[["geometry"]]],
        ignore_index=True,
    ),
    crs=current_station_hillingdon.crs,
)

print(f"Demand points: {len(demand_hillingdon)}")
print(f"Current stations: {len(current_station_hillingdon)}")
print(f"Candidate stations: {len(candidate_station_hillingdon)}")
print(f"Total candidate stations: {len(total_station_hillingdon)} (current + new)")
print(f"CRS: {demand_hillingdon.crs}, {current_station_hillingdon.crs}, {candidate_station_hillingdon.crs}, {total_station_hillingdon.crs}")

## 3.2 Distance and coverage

We compute Euclidean distances in metres and a 3 km coverage matrix.


In [ ]:
demand_coords = np.array([(p.x, p.y) for p in demand_hillingdon.geometry])
total_station_coords = np.array([(p.x, p.y) for p in total_station_hillingdon.geometry])

n_demand = len(demand_hillingdon)
n_current_station = len(current_station_hillingdon)
n_total_station = len(total_station_hillingdon)

dist_demand_total_station = np.sqrt(((demand_coords[:, None, :] - total_station_coords[None, :, :]) ** 2).sum(axis=2))
coverage_demand_total_station = (dist_demand_total_station <= 3000).astype(int)
coverage_demand_current_station = coverage_demand_total_station[:, :n_current_station]

print(f"Current 3 km coverage: {(coverage_demand_current_station.sum(axis=1) > 0).sum() / n_demand * 100:.2f}%")
print(f"Uncovered demand points: {(coverage_demand_current_station.sum(axis=1) == 0).sum()}")

## 3.3 Model construction

We build the optimisation model here. It is solved in two steps:

1. **Minimise the number of stations** needed to achieve full 3 km coverage, which gives the lower bound `pmin`.
2. **Minimise the fire-risk-weighted travel distance** subject to a station-count limit `p` (at `pmin`, or any larger feasible `p`).

An optional retention parameter `q` can require that exactly `q` existing stations remain open.

In [ ]:
weights = demand_hillingdon["fire_risk"].values

def solve_min_stations(q=None):
    """Minimise the number of open stations subject to 3 km coverage."""
    prob = pulp.LpProblem("MinFireStations", pulp.LpMinimize)
    Y = pulp.LpVariable.dicts("Y", range(n_total_station), cat="Binary")
    X = pulp.LpVariable.dicts(
        "X",
        ((i, j) for i in range(n_demand) for j in range(n_total_station)),
        lowBound=0,
        upBound=1,
        cat="Continuous",
    )
    prob += pulp.lpSum(Y[j] for j in range(n_total_station))

    for i in range(n_demand):
        prob += pulp.lpSum(X[i, j] for j in range(n_total_station)) == 1
        for j in range(n_total_station):
            if coverage_demand_total_station[i, j] == 1:
                prob += X[i, j] <= Y[j]
            else:
                prob += X[i, j] == 0

    if q is not None:
        prob += pulp.lpSum(
            Y[j] for j in range(n_current_station)
        ) == q

    prob.solve()
    print("Status:", pulp.LpStatus[prob.status])
    print("Minimum station count pmin =", pulp.value(prob.objective))
    return pulp.value(prob.objective)


def solve_min_weighted_distance(p, q=None, exact=False):
    """Minimise risk-weighted distance with at most (or exactly) p stations."""
    prob = pulp.LpProblem("MinWeightedDistance", pulp.LpMinimize)
    Y = pulp.LpVariable.dicts("Y", range(n_total_station), cat="Binary")
    X = pulp.LpVariable.dicts(
        "X",
        ((i, j) for i in range(n_demand) for j in range(n_total_station)),
        lowBound=0,
        upBound=1,
        cat="Continuous",
    )
    prob += pulp.lpSum(
        weights[i] * dist_demand_total_station[i, j] * X[i, j]
        for i in range(n_demand)
        for j in range(n_total_station)
    )

    for i in range(n_demand):
        prob += pulp.lpSum(X[i, j] for j in range(n_total_station)) == 1
        for j in range(n_total_station):
            if coverage_demand_total_station[i, j] == 1:
                prob += X[i, j] <= Y[j]
            else:
                prob += X[i, j] == 0

    if q is not None:
        prob += pulp.lpSum(
            Y[j] for j in range(n_current_station)
        ) == q

    if exact:
        prob += pulp.lpSum(Y[j] for j in range(n_total_station)) == p
    else:
        prob += pulp.lpSum(Y[j] for j in range(n_total_station)) <= p

    prob.solve()
    selected_sites = [j for j in range(n_total_station) if pulp.value(Y[j]) > 0.5]
    print("Status:", pulp.LpStatus[prob.status])
    print("Weighted distance =", pulp.value(prob.objective))
    print("Selected sites:", selected_sites)
    return selected_sites, pulp.value(prob.objective)

## 3.4 Scenarios by retained current stations

We run the model under five scenarios with q=0,1,2,3,4: exactly that many existing stations remain open.


In [ ]:
q_values = [0, 1, 2, 3, 4]
scenario_results = {}

for q in q_values:
    pmin = solve_min_stations(q=q)
    selected, obj = solve_min_weighted_distance(pmin, q=q)
    scenario_results[q] = {
        "pmin": pmin,
        "selected_sites": selected,
        "objective_value": obj,
        "n_stations": len(selected),
    }
    print(f"q={q}: {len(selected)} stations, pmin={pmin}, objective={obj}")


In [ ]:
for q, res in scenario_results.items():
    print(f"q={q}: n={res['n_stations']}, pmin={res['pmin']}, obj={res['objective_value']}")

## 3.5 Pareto front

For each scenario, we vary the total station count from pmin and record the minimum weighted distance.


In [ ]:
pareto_results_by_q = {}
n_candidate_station = n_total_station - n_current_station  # MSOA candidates

for q in q_values:
    p_min = int(scenario_results[q]["pmin"])
    p_max = q + n_candidate_station  # q current + all MSOA
    pareto_results = []
    for p in range(p_min, p_max + 1):
        selected_sites, objective_value = solve_min_weighted_distance(
            p, q=q, exact=True
        )
        pareto_results.append(
            {"p": p, "selected_sites": selected_sites, "objective_value": objective_value}
        )
    pareto_results_by_q[q] = pareto_results
    print(f"q={q}: {len(pareto_results)} solutions (p={p_min} to {p_max})")


In [ ]:
colors = ["#b5aed5", "#b2e6fd", "#b8d2cc", "#e8b2a7", "#feebb9"]
x_middle = [12, 16, 20, 24, 28]
x_middle_with_32 = x_middle + [32]

fig, axes = plt.subplots(2, 3, figsize=(7.2, 6), dpi=400)
axes = axes.flatten()

# panel 1: all q together
ax = axes[0]
for q, color in zip(q_values, colors):
    results = pareto_results_by_q[q]
    x = [res["p"] for res in results]
    y = [res["objective_value"] for res in results]
    ax.plot(
        x, y, marker="o", markersize=1.5, linewidth=0.5,
        linestyle="-", color=color, alpha=0.6, label=f"q={q}",
    )
ax.set_title("All scenarios", fontsize=8)
legend_handles = [
    Line2D(
        [0], [0], color=color, marker="o", markersize=1.5,
        linewidth=0.5, alpha=1.0, label=f"q={q}",
    )
    for q, color in zip(q_values, colors)
]
leg = ax.legend(handles=legend_handles, fontsize=5, loc="best")
leg.get_frame().set_linewidth(0.2)
ax.set_xlim(7, 37)
ax.set_xticks([8] + x_middle_with_32 + [36])

# panels 2-6: shared middle ticks; ends are each series pmin / pmax
# only All scenarios and q=4 include x=32 in the middle ticks
for ax, q, color in zip(axes[1:], q_values, colors):
    results = pareto_results_by_q[q]
    x = [res["p"] for res in results]
    y = [res["objective_value"] for res in results]
    ax.plot(
        x, y, marker="o", markersize=2, linewidth=1,
        linestyle="-", color=color, alpha=1.0, label=f"q={q}",
    )
    ax.set_title(f"q={q}", fontsize=8)
    p_min, p_max = x[0], x[-1]
    middle = x_middle_with_32 if q == 4 else x_middle
    ax.set_xlim(7, 37)
    ax.set_xticks(sorted(set([p_min, p_max] + middle)))

for ax in axes:
    ax.set_xlabel("Number of fire stations", fontsize=6)
    ax.set_ylabel("Total weighted distance", fontsize=6)
    ax.set_ylim(1.28e7, 2.85e7)
    ax.set_yticks(np.arange(1.4e7, 2.81e7, 0.1e7))
    ax.tick_params(axis="both", labelsize=5)
    ax.yaxis.get_offset_text().set_fontsize(5)
    ax.grid(True, axis="y", linestyle="--", linewidth=0.4, alpha=0.35)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

fig.tight_layout()
os.makedirs("figure", exist_ok=True)
fig.savefig("figure/pareto_front.png", dpi=400)
plt.show()


## 3.6 Site visualisation

Map the chosen solution.


In [ ]:
# choose which solution to map
q_retain = 4
p_show = 10  # must be between pmin and p_max for this q

selected_sites = next(
    res["selected_sites"]
    for res in pareto_results_by_q[q_retain]
    if res["p"] == p_show
)
retained_current = total_station_hillingdon.iloc[
    [j for j in selected_sites if j < n_current_station]
].copy()
new_stations = total_station_hillingdon.iloc[
    [j for j in selected_sites if j >= n_current_station]
].copy()

h_fire_risk = oa[oa.geometry.centroid.within(hillingdon_geom)].copy()
hillingdon = borough[borough["GSS_CODE"] == "E09000017"].copy()

fig, ax = plt.subplots(figsize=(7.2, 6), dpi=400)
fig.subplots_adjust(left=0.02, right=0.98, bottom=0.02, top=0.98)

# Hillingdon-focused layers (same styling as the study-area map)
hillingdon.boundary.plot(
    edgecolor="black", linewidth=0.45, alpha=0.45, ax=ax
)
h_fire_risk.plot(
    column="fire_risk", cmap=cmap, legend=False, alpha=0.6, ax=ax
)

for _, station in pd.concat([retained_current, new_stations], ignore_index=True).iterrows():
    gpd.GeoSeries([station.geometry.buffer(3000)], crs=current_station_hillingdon.crs).plot(
        edgecolor="black", facecolor="none", linestyle="--", linewidth=0.7, alpha=0.7, ax=ax
    )

retained_current.plot(color="#9ACD32", markersize=6, ax=ax)
new_stations.plot(color="#FFD700", markersize=6, ax=ax)

finish_fire_risk_map(
    fig,
    ax,
    extra_legend_handles=[
        Line2D(
            [0], [0], color="#FFD700", marker="o", linestyle="None",
            markersize=3, label="New Fire Station",
        )
    ],
    save_path="figure/selected_stations.png",
)